# 🗄️ SQL con Python: sqlite3, SQLAlchemy y Pandas

**Módulo:** Integración SQL — Ejercicios Integradores  
**Nivel:** Intermedio

---

## 📋 ¿Qué vas a aprender?

| Bloque | Temas |
|--------|-------|
| 🔌 sqlite3 | Crear BD, tablas relacionadas, insertar y consultar datos |
| 🏗️ SQLAlchemy (Core) | Conexión, tablas declarativas, consultas con ORM |
| 🏗️ SQLAlchemy (ORM) | Modelos como clases Python, sesiones, relaciones |
| 🐼 SQLAlchemy + Pandas | Leer SQL directo a DataFrame, analizar con pandas |
| 📊 Proyecto integrador | Pipeline completo: BD → SQL → Pandas → análisis |

---

> 💡 **¿Por qué SQLite?**  
> SQLite no requiere instalar ningún servidor. Toda la base de datos vive en un archivo `.db`  
> (o en memoria con `:memory:`). Es ideal para aprender, prototipar y aplicaciones pequeñas.

> **Instalación:**
> ```bash
> pip install sqlalchemy pandas
> ```
> `sqlite3` ya viene incluido en Python estándar — no necesita instalación.

---
## 🔌 Bloque 1: sqlite3 — Libros y Autores

### 📘 Teoría

`sqlite3` es el módulo estándar de Python para trabajar con bases de datos SQLite.

#### Flujo de trabajo básico
```python
import sqlite3

# 1. Conectar (crea el archivo si no existe)
conn   = sqlite3.connect('mi_base.db')  # o ':memory:' para BD en RAM
cursor = conn.cursor()

# 2. Ejecutar SQL
cursor.execute('CREATE TABLE ...')
cursor.execute('INSERT INTO ... VALUES (?, ?, ?)', (val1, val2, val3))
cursor.executemany('INSERT INTO ...', lista_de_tuplas)

# 3. Guardar cambios
conn.commit()

# 4. Consultar
cursor.execute('SELECT ...')
filas = cursor.fetchall()   # lista de tuplas
fila  = cursor.fetchone()   # una sola tupla

# 5. Cerrar
conn.close()
```

#### Context manager (forma recomendada)
```python
with sqlite3.connect('mi_base.db') as conn:
    cursor = conn.cursor()
    cursor.execute('...')
    # conn.commit() es automático al salir del with (si no hubo error)
```

#### Relaciones entre tablas
```sql
-- La clave foránea conecta libros con autores
CREATE TABLE autores (
    id     INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT    NOT NULL
);

CREATE TABLE libros (
    id        INTEGER PRIMARY KEY AUTOINCREMENT,
    titulo    TEXT    NOT NULL,
    autor_id  INTEGER REFERENCES autores(id),  -- FK
    anio      INTEGER
);
```

### 💡 Ejemplo resuelto 1 — Base de datos de libros y autores

In [ ]:
import sqlite3

# ✅ Crear una BD relacional de libros y autores

with sqlite3.connect(':memory:') as conn:
    conn.execute('PRAGMA foreign_keys = ON')
    cursor = conn.cursor()

    # ── Crear tablas ──
    cursor.executescript("""
        CREATE TABLE autores (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            nombre      TEXT    NOT NULL,
            nacionalidad TEXT,
            anio_nac    INTEGER
        );

        CREATE TABLE libros (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            titulo      TEXT    NOT NULL,
            autor_id    INTEGER NOT NULL REFERENCES autores(id),
            anio_pub    INTEGER,
            genero      TEXT,
            paginas     INTEGER
        );
    """)

    # ── Insertar autores ──
    autores_data = [
        ('Jorge Luis Borges',        'Argentina', 1899),
        ('Julio Cortázar',           'Argentina', 1914),
        ('Gabriel García Márquez',   'Colombia',  1927),
        ('Isabel Allende',           'Chile',     1942),
        ('Mario Vargas Llosa',       'Perú',      1936),
    ]
    cursor.executemany(
        'INSERT INTO autores (nombre, nacionalidad, anio_nac) VALUES (?, ?, ?)',
        autores_data
    )

    # ── Insertar libros ──
    libros_data = [
        ('Ficciones',               1, 1944, 'Cuentos',   174),
        ('El Aleph',                1, 1949, 'Cuentos',   203),
        ('El libro de arena',       1, 1975, 'Cuentos',   187),
        ('Rayuela',                 2, 1963, 'Novela',    635),
        ('Bestiario',               2, 1951, 'Cuentos',   166),
        ('Historias de Cronopios',  2, 1962, 'Cuentos',   120),
        ('Cien años de soledad',    3, 1967, 'Novela',    432),
        ('El amor en los tiempos', 3, 1985, 'Novela',    444),
        ('La casa de los espíritus',4, 1982, 'Novela',    368),
        ('La ciudad y los perros',  5, 1963, 'Novela',    357),
    ]
    cursor.executemany(
        'INSERT INTO libros (titulo, autor_id, anio_pub, genero, paginas) VALUES (?, ?, ?, ?, ?)',
        libros_data
    )
    conn.commit()

    # ── Consultas ──
    print('=== Todos los libros con su autor ===')
    query = """
        SELECT a.nombre, l.titulo, l.anio_pub, l.genero
        FROM libros l
        JOIN autores a ON l.autor_id = a.id
        ORDER BY a.nombre, l.anio_pub
    """
    for row in cursor.execute(query):
        print(f'  {row[0]:28} | {row[2]} | {row[3]:8} | {row[1]}')

    print('\n=== Libros por autor (conteo y promedio páginas) ===')
    query2 = """
        SELECT a.nombre,
               COUNT(l.id)        AS cantidad,
               AVG(l.paginas)     AS prom_paginas,
               MIN(l.anio_pub)    AS primer_libro
        FROM autores a
        JOIN libros l ON a.id = l.autor_id
        GROUP BY a.nombre
        ORDER BY cantidad DESC
    """
    print(f'  {"Autor":28} | Libros | Pág. prom | Desde')
    print('  ' + '-'*58)
    for row in cursor.execute(query2):
        print(f'  {row[0]:28} | {row[1]:6} | {row[2]:9.0f} | {row[3]}')

    print('\n=== Autores argentinos con más de 2 libros ===')
    query3 = """
        SELECT a.nombre, COUNT(*) as libros
        FROM autores a
        JOIN libros l ON a.id = l.autor_id
        WHERE a.nacionalidad = 'Argentina'
        GROUP BY a.nombre
        HAVING COUNT(*) > 2
    """
    for row in cursor.execute(query3):
        print(f'  {row[0]} — {row[1]} libros')

### ✏️ Ejercicio guiado 1 — Ampliar el esquema con editoriales

Extendé la BD anterior agregando una tabla `editoriales` y vinculándola con `libros`.

**Nueva estructura:**
```
autores ──< libros >── editoriales
```

**Pistas:**
- Agregá `editorial_id` como FK en la tabla `libros`
- Usá `LEFT JOIN` para mostrar libros aunque no tengan editorial asignada
- `COALESCE(e.nombre, 'Sin editorial')` muestra un valor por defecto si es NULL

In [ ]:
import sqlite3

with sqlite3.connect(':memory:') as conn:
    conn.execute('PRAGMA foreign_keys = ON')
    cursor = conn.cursor()

    cursor.executescript("""
        CREATE TABLE autores (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            nombre TEXT NOT NULL, nacionalidad TEXT
        );
        CREATE TABLE editoriales (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            nombre TEXT NOT NULL,
            pais   TEXT
        );
        CREATE TABLE libros (
            id           INTEGER PRIMARY KEY AUTOINCREMENT,
            titulo       TEXT    NOT NULL,
            autor_id     INTEGER REFERENCES autores(id),
            editorial_id INTEGER REFERENCES editoriales(id),
            anio_pub     INTEGER,
            genero       TEXT
        );
    """)

    # Insertar datos base
    cursor.executemany('INSERT INTO autores (nombre, nacionalidad) VALUES (?, ?)', [
        ('Jorge Luis Borges', 'Argentina'),
        ('Julio Cortázar',    'Argentina'),
        ('Gabriel García Márquez', 'Colombia'),
    ])
    cursor.executemany('INSERT INTO editoriales (nombre, pais) VALUES (?, ?)', [
        ('Sudamericana', 'Argentina'),
        ('Alfaguara',    'España'),
        ('Seix Barral',  'España'),
    ])
    cursor.executemany(
        'INSERT INTO libros (titulo, autor_id, editorial_id, anio_pub, genero) VALUES (?, ?, ?, ?, ?)',
        [
            ('Ficciones',              1, 1, 1944, 'Cuentos'),
            ('El Aleph',               1, 2, 1949, 'Cuentos'),
            ('Rayuela',                2, 1, 1963, 'Novela'),
            ('Bestiario',              2, None, 1951, 'Cuentos'),  # sin editorial
            ('Cien años de soledad',   3, 3, 1967, 'Novela'),
        ]
    )
    conn.commit()

    # ✏️ Consulta 1: todos los libros con autor y editorial (LEFT JOIN para no perder los sin editorial)
    print('=== Libros con autor y editorial ===')
    # Tu consulta SQL aquí:


    # ✏️ Consulta 2: editoriales y cuántos libros publicaron
    print('\n=== Libros por editorial ===')
    # Tu consulta SQL aquí:


    # ✏️ Consulta 3: libros SIN editorial asignada
    print('\n=== Libros sin editorial ===')
    # Pista: WHERE l.editorial_id IS NULL


<details>
<summary>👀 Ver solución</summary>

```python
# Consulta 1
cursor.execute("""
    SELECT a.nombre, l.titulo, COALESCE(e.nombre, 'Sin editorial'), l.anio_pub
    FROM libros l
    JOIN autores a ON l.autor_id = a.id
    LEFT JOIN editoriales e ON l.editorial_id = e.id
    ORDER BY a.nombre, l.anio_pub
""")
for row in cursor.fetchall():
    print(f'  {row[0]:25} | {row[1]:25} | {row[2]:15} | {row[3]}')

# Consulta 2
cursor.execute("""
    SELECT e.nombre, COUNT(l.id) as cant
    FROM editoriales e
    LEFT JOIN libros l ON e.id = l.editorial_id
    GROUP BY e.nombre ORDER BY cant DESC
""")

# Consulta 3
cursor.execute('SELECT titulo FROM libros WHERE editorial_id IS NULL')
```
</details>

### 🚀 Ejercicio libre 1 — Biblioteca completa

Diseñá e implementá una BD de biblioteca con las siguientes tablas:
- `autores`: id, nombre, nacionalidad, anio_nac
- `editoriales`: id, nombre, pais, fundacion
- `libros`: id, titulo, autor_id, editorial_id, anio_pub, genero, paginas, isbn
- `socios`: id, nombre, email, ciudad
- `prestamos`: id, socio_id, libro_id, fecha_prestamo, fecha_devolucion

Insertá datos reales (al menos 5 autores, 3 editoriales, 10 libros, 4 socios, 8 préstamos).

Respondé con consultas SQL:
1. ¿Qué socio tiene más préstamos activos (sin fecha de devolución)?
2. ¿Qué libro fue prestado más veces?
3. ¿Cuántos días en promedio dura un préstamo? (`julianday(devolucion) - julianday(prestamo)`)
4. Libros de autores argentinos publicados después de 1950, ordenados por año

In [ ]:
import sqlite3

# 🚀 Tu solución aquí:


---
## 🏗️ Bloque 2: SQLAlchemy — Core y ORM

### 📘 Teoría

**SQLAlchemy** es la biblioteca de acceso a bases de datos más completa de Python.  
Tiene dos capas de uso:

| Capa | Qué es | Cuándo usarla |
|------|--------|---------|
| **Core** | SQL expresivo en Python | Cuando querés control total sobre las queries |
| **ORM** | Tablas como clases Python | Cuando querés trabajar con objetos, no SQL crudo |

#### SQLAlchemy Core — Conceptos clave
```python
from sqlalchemy import create_engine, text

# Motor de conexión (SQLite en archivo)
engine = create_engine('sqlite:///mi_base.db')

# Ejecutar SQL directo
with engine.connect() as conn:
    resultado = conn.execute(text('SELECT * FROM libros'))
    for fila in resultado:
        print(fila)
```

#### SQLAlchemy ORM — Definir modelos
```python
from sqlalchemy import Column, Integer, String, ForeignKey
from sqlalchemy.orm import DeclarativeBase, relationship

class Base(DeclarativeBase): pass

class Autor(Base):
    __tablename__ = 'autores'
    id     = Column(Integer, primary_key=True)
    nombre = Column(String, nullable=False)
    libros = relationship('Libro', back_populates='autor')  # 1:N

class Libro(Base):
    __tablename__ = 'libros'
    id       = Column(Integer, primary_key=True)
    titulo   = Column(String, nullable=False)
    autor_id = Column(Integer, ForeignKey('autores.id'))
    autor    = relationship('Autor', back_populates='libros')
```

### 💡 Ejemplo resuelto 2 — SQLAlchemy ORM: modelos, sesión y consultas

In [ ]:
from sqlalchemy import (
    create_engine, Column, Integer, String, Float,
    ForeignKey, Date, Boolean, text
)
from sqlalchemy.orm import DeclarativeBase, relationship, Session
from datetime import date

# ✅ Sistema de ventas con SQLAlchemy ORM

# ── Definir modelos ──
class Base(DeclarativeBase): pass

class Categoria(Base):
    __tablename__ = 'categorias'
    id     = Column(Integer, primary_key=True, autoincrement=True)
    nombre = Column(String(50), nullable=False, unique=True)
    productos = relationship('Producto', back_populates='categoria')

    def __repr__(self):
        return f'<Categoria({self.nombre})>'

class Producto(Base):
    __tablename__ = 'productos'
    id           = Column(Integer, primary_key=True, autoincrement=True)
    nombre       = Column(String(100), nullable=False)
    precio       = Column(Float, nullable=False)
    stock        = Column(Integer, default=0)
    categoria_id = Column(Integer, ForeignKey('categorias.id'))
    categoria    = relationship('Categoria', back_populates='productos')
    items        = relationship('ItemVenta', back_populates='producto')

    def __repr__(self):
        return f'<Producto({self.nombre}, ${self.precio})>'

class Venta(Base):
    __tablename__ = 'ventas'
    id     = Column(Integer, primary_key=True, autoincrement=True)
    fecha  = Column(String, default=lambda: date.today().isoformat())
    items  = relationship('ItemVenta', back_populates='venta')

class ItemVenta(Base):
    __tablename__ = 'items_venta'
    id          = Column(Integer, primary_key=True, autoincrement=True)
    venta_id    = Column(Integer, ForeignKey('ventas.id'))
    producto_id = Column(Integer, ForeignKey('productos.id'))
    cantidad    = Column(Integer, nullable=False)
    precio_unit = Column(Float,   nullable=False)
    venta       = relationship('Venta',    back_populates='items')
    producto    = relationship('Producto', back_populates='items')

    @property
    def subtotal(self):
        return self.cantidad * self.precio_unit


# ── Crear BD en memoria ──
engine = create_engine('sqlite:///:memory:', echo=False)
Base.metadata.create_all(engine)

# ── Insertar datos con sesión ──
with Session(engine) as session:
    # Crear categorías
    cats = {
        'Electrónica': Categoria(nombre='Electrónica'),
        'Ropa':        Categoria(nombre='Ropa'),
        'Hogar':       Categoria(nombre='Hogar'),
    }
    session.add_all(cats.values())

    # Crear productos
    prods = [
        Producto(nombre='Laptop Pro',  precio=1200.0, stock=15, categoria=cats['Electrónica']),
        Producto(nombre='Auriculares', precio=89.99,  stock=50, categoria=cats['Electrónica']),
        Producto(nombre='Camiseta',    precio=25.00,  stock=100,categoria=cats['Ropa']),
        Producto(nombre='Pantalón',    precio=60.00,  stock=80, categoria=cats['Ropa']),
        Producto(nombre='Sillón',      precio=350.00, stock=8,  categoria=cats['Hogar']),
    ]
    session.add_all(prods)

    # Crear ventas
    v1 = Venta(fecha='2024-01-10')
    v1.items = [
        ItemVenta(producto=prods[0], cantidad=1, precio_unit=1200.0),
        ItemVenta(producto=prods[1], cantidad=2, precio_unit=89.99),
    ]
    v2 = Venta(fecha='2024-01-15')
    v2.items = [
        ItemVenta(producto=prods[2], cantidad=3, precio_unit=25.00),
        ItemVenta(producto=prods[4], cantidad=1, precio_unit=350.00),
    ]
    v3 = Venta(fecha='2024-02-01')
    v3.items = [
        ItemVenta(producto=prods[0], cantidad=2, precio_unit=1150.0),
        ItemVenta(producto=prods[3], cantidad=5, precio_unit=60.00),
    ]
    session.add_all([v1, v2, v3])
    session.commit()

    # ── Consultas ORM ──
    print('=== Todos los productos por categoría ===')
    for cat in session.query(Categoria).order_by(Categoria.nombre):
        print(f'  📁 {cat.nombre}')
        for p in cat.productos:
            print(f'     - {p.nombre:15} ${p.precio:>8.2f}  stock: {p.stock}')

    print('\n=== Todas las ventas con subtotales ===')
    for venta in session.query(Venta).order_by(Venta.fecha):
        total = sum(item.subtotal for item in venta.items)
        print(f'  Venta #{venta.id} ({venta.fecha}) — Total: ${total:.2f}')
        for item in venta.items:
            print(f'    · {item.producto.nombre:15} x{item.cantidad}  ${item.subtotal:.2f}')

    # ── Consulta con SQL directo (text) ──
    print('\n=== Ingreso total por categoría (SQL directo) ===')
    resultado = session.execute(text("""
        SELECT c.nombre, SUM(iv.cantidad * iv.precio_unit) as total
        FROM categorias c
        JOIN productos p ON c.id = p.categoria_id
        JOIN items_venta iv ON p.id = iv.producto_id
        GROUP BY c.nombre
        ORDER BY total DESC
    """))
    for row in resultado:
        print(f'  {row[0]:15} | ${row[1]:,.2f}')

### ✏️ Ejercicio guiado 2 — CRUD completo con ORM

Usando SQLAlchemy ORM, implementá las operaciones CRUD sobre la BD de ventas del ejemplo.

**Pistas:**
- **Crear:** `session.add(objeto)` + `session.commit()`
- **Leer:** `session.query(Modelo).filter_by(campo=valor).first()`
- **Actualizar:** modificá el atributo del objeto + `session.commit()`
- **Eliminar:** `session.delete(objeto)` + `session.commit()`

In [ ]:
# (Requiere haber ejecutado la celda anterior)

with Session(engine) as session:

    # ✏️ CREATE: agregá un producto nuevo llamado 'Tablet 10"' en categoría Electrónica
    print('=== CREATE: nuevo producto ===')
    # Pista: primero buscá la categoría, luego creá el producto
    # cat_elec = session.query(Categoria).filter_by(nombre='Electrónica').first()
    # nuevo_prod = Producto(...)


    # ✏️ READ: listar todos los productos con precio > 100
    print('\n=== READ: productos con precio > $100 ===')
    # Pista: session.query(Producto).filter(Producto.precio > 100).all()


    # ✏️ UPDATE: actualizá el stock de 'Auriculares' a 75
    print('\n=== UPDATE: cambiar stock de Auriculares ===')
    # Pista: buscá el producto, modificá .stock, hacé commit


    # ✏️ DELETE: eliminá el producto 'Pantalón'
    print('\n=== DELETE: eliminar Pantalón ===')
    # Pista: session.delete(producto) + commit


    # Verificación final
    print('\n=== Estado final de productos ===')
    for p in session.query(Producto).order_by(Producto.nombre):
        print(f'  {p.nombre:15} ${p.precio:>8.2f}  stock: {p.stock}')


<details>
<summary>👀 Ver solución</summary>

```python
# CREATE
cat_elec = session.query(Categoria).filter_by(nombre='Electrónica').first()
nuevo_prod = Producto(nombre='Tablet 10"', precio=299.99, stock=20, categoria=cat_elec)
session.add(nuevo_prod)
session.commit()
print(f'  Creado: {nuevo_prod.nombre} (id={nuevo_prod.id})')

# READ
caros = session.query(Producto).filter(Producto.precio > 100).all()
for p in caros: print(f'  {p.nombre} — ${p.precio}')

# UPDATE
auri = session.query(Producto).filter_by(nombre='Auriculares').first()
auri.stock = 75
session.commit()
print(f'  Stock actualizado: {auri.nombre} → {auri.stock}')

# DELETE
pantalon = session.query(Producto).filter_by(nombre='Pantalón').first()
session.delete(pantalon)
session.commit()
print('  Pantalón eliminado')
```
</details>

### 🚀 Ejercicio libre 2 — Sistema de gestión escolar

Implementá un sistema escolar completo con SQLAlchemy ORM:

**Modelos:**
- `Alumno`: id, nombre, email, fecha_nacimiento
- `Profesor`: id, nombre, especialidad
- `Curso`: id, nombre, descripcion, profesor_id (FK)
- `Inscripcion`: id, alumno_id, curso_id, fecha, nota_final (puede ser NULL)

**Relaciones:**
- Un `Profesor` dicta muchos `Curso` (1:N)
- Un `Alumno` puede estar en muchos `Curso` y viceversa (N:M via `Inscripcion`)

**Implementá:**
1. Todos los modelos con `relationship` bidireccional
2. Insertá al menos 3 profesores, 5 cursos, 6 alumnos y 10 inscripciones
3. CRUD completo: crear alumno, leer inscripciones de un curso, actualizar nota, eliminar inscripción
4. Consulta: promedio de notas por curso (usando `text()` o ORM)

In [ ]:
from sqlalchemy import create_engine, Column, Integer, String, Float, ForeignKey
from sqlalchemy.orm import DeclarativeBase, relationship, Session

# 🚀 Tu solución aquí:


---
## 🐼 Bloque 3: SQLAlchemy + Pandas — Análisis de Datos

### 📘 Teoría

Pandas puede leer directamente desde una base de datos SQL usando `pd.read_sql()`.  
Combinado con SQLAlchemy, esto crea un pipeline muy poderoso:

```
Base de datos  →  SQLAlchemy  →  pd.read_sql()  →  DataFrame  →  Análisis
```

#### Funciones clave
```python
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine('sqlite:///ventas.db')

# Leer tabla completa
df = pd.read_sql('SELECT * FROM ventas', engine)

# Leer con query compleja
df = pd.read_sql("""
    SELECT v.fecha, c.nombre as categoria, SUM(iv.cantidad * iv.precio_unit) as total
    FROM ventas v
    JOIN items_venta iv ON v.id = iv.venta_id
    JOIN productos p    ON iv.producto_id = p.id
    JOIN categorias c   ON p.categoria_id = c.id
    GROUP BY v.fecha, c.nombre
""", engine)

# Escribir DataFrame a SQL
df_nuevo.to_sql('tabla_resultado', engine, if_exists='replace', index=False)
```

#### Parámetros útiles de `read_sql`
| Parámetro | Descripción |
|-----------|-------------|
| `parse_dates=['fecha']` | Convierte columnas a datetime automáticamente |
| `index_col='id'` | Usa una columna como índice del DataFrame |
| `chunksize=1000` | Lee en chunks (útil para tablas enormes) |

### 💡 Ejemplo resuelto 3 — Pipeline SQL → Pandas → Análisis de ventas

In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from sqlalchemy.orm import Session

# ✅ Pipeline completo: poblar BD → leer con Pandas → analizar

# ── 1. Crear y poblar la BD ──
engine = create_engine('sqlite:///:memory:', echo=False)

# Usamos SQL directo para poblar rápidamente
with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE categorias (id INTEGER PRIMARY KEY, nombre TEXT);
        CREATE TABLE productos  (id INTEGER PRIMARY KEY, nombre TEXT, precio REAL, categoria_id INTEGER);
        CREATE TABLE ventas     (id INTEGER PRIMARY KEY, fecha TEXT, region TEXT);
        CREATE TABLE items      (id INTEGER PRIMARY KEY, venta_id INTEGER, producto_id INTEGER, cantidad INTEGER, precio_unit REAL);
    """))

    conn.execute(text("INSERT INTO categorias VALUES (1,'Electrónica'),(2,'Ropa'),(3,'Hogar'),(4,'Deportes')"))
    conn.execute(text("""
        INSERT INTO productos VALUES
        (1,'Laptop',1200,1),(2,'Mouse',25,1),(3,'Auriculares',89,1),
        (4,'Camiseta',25,2),(5,'Zapatillas',120,4),(6,'Sillón',350,3)
    """))

    # Generar 200 ventas sintéticas
    np.random.seed(42)
    fechas  = pd.date_range('2023-01-01', '2023-12-31', periods=200)
    regiones = ['AMBA', 'Centro', 'NOA', 'Patagonia']
    ventas_rows = [(i+1, str(f.date()), np.random.choice(regiones)) for i, f in enumerate(fechas)]
    conn.execute(text('INSERT INTO ventas VALUES (:id, :fecha, :region)'),
                 [{'id': r[0], 'fecha': r[1], 'region': r[2]} for r in ventas_rows])

    items_rows = []
    item_id = 1
    for venta_id, _, _ in ventas_rows:
        n_items = np.random.randint(1, 4)
        prod_ids = np.random.choice([1,2,3,4,5,6], n_items, replace=False)
        for pid in prod_ids:
            qty = int(np.random.randint(1, 6))
            # precio con variación ±10%
            precios_base = {1:1200,2:25,3:89,4:25,5:120,6:350}
            precio = round(precios_base[pid] * np.random.uniform(0.9, 1.1), 2)
            items_rows.append({'id': item_id, 'venta_id': venta_id, 'producto_id': pid, 'cantidad': qty, 'precio_unit': precio})
            item_id += 1
    conn.execute(text('INSERT INTO items VALUES (:id, :venta_id, :producto_id, :cantidad, :precio_unit)'), items_rows)

# ── 2. Leer con Pandas ──
query_ventas = """
    SELECT
        v.fecha,
        v.region,
        c.nombre  AS categoria,
        p.nombre  AS producto,
        i.cantidad,
        i.precio_unit,
        i.cantidad * i.precio_unit AS ingreso
    FROM ventas v
    JOIN items i       ON v.id = i.venta_id
    JOIN productos p   ON i.producto_id = p.id
    JOIN categorias c  ON p.categoria_id = c.id
"""
df = pd.read_sql(query_ventas, engine, parse_dates=['fecha'])
df['mes'] = df['fecha'].dt.to_period('M')

print(f'Dataset cargado: {df.shape[0]} registros')
print(f'Período: {df["fecha"].min().date()} → {df["fecha"].max().date()}')
print(f'Ingreso total: ${df["ingreso"].sum():,.0f}\n')

# ── 3. Análisis con Pandas ──
print('=== Ingreso por categoría ===')
por_cat = df.groupby('categoria')['ingreso'].agg(['sum','mean','count'])
por_cat.columns = ['total', 'ticket_avg', 'transacciones']
por_cat['total'] = por_cat['total'].round(0)
por_cat['pct'] = (por_cat['total'] / por_cat['total'].sum() * 100).round(1)
print(por_cat.sort_values('total', ascending=False))

print('\n=== Ingreso mensual por región ===')
pivot = df.pivot_table(
    values='ingreso', index='mes',
    columns='region', aggfunc='sum'
).round(0)
print(pivot.tail(6))  # últimos 6 meses

print('\n=== Top 3 productos por región ===')
top_prod = (
    df.groupby(['region', 'producto'])['ingreso']
    .sum()
    .reset_index()
    .sort_values(['region', 'ingreso'], ascending=[True, False])
    .groupby('region')
    .head(3)
)
print(top_prod.to_string(index=False))

### ✏️ Ejercicio guiado 3 — Análisis de período y exportación

Usando el `engine` y el DataFrame `df` del ejemplo anterior, completá los análisis.

**Pistas:**
- `pd.read_sql(query_con_parametros, engine, params={'fecha': '2023-06-01'})` para pasar parámetros
- `df.to_sql('nombre_tabla', engine, if_exists='replace', index=False)` para guardar resultados en la BD
- `df.resample('Q', on='fecha')['ingreso'].sum()` agrupa por trimestre

In [ ]:
# (Requiere haber ejecutado la celda anterior — engine y df disponibles)

# ✏️ Análisis 1: ventas del Q3 2023 (julio-septiembre)
print('=== Ventas Q3 2023 ===')
# Pista: filtrá df por fecha >= '2023-07-01' y < '2023-10-01'


# ✏️ Análisis 2: variación mensual del ingreso total (mes a mes)
print('\n=== Variación mensual ===')
# Pista: groupby('mes')['ingreso'].sum() y luego .pct_change()


# ✏️ Análisis 3: exportar resumen por categoría+mes a una nueva tabla SQL
print('\n=== Exportar resumen a tabla SQL ===')
# Pista: df.groupby(['mes','categoria']).sum().reset_index() luego .to_sql()


# ✏️ Verificar que la tabla se creó leyéndola de vuelta
print('\n=== Verificar tabla exportada ===')
# Pista: pd.read_sql('SELECT * FROM resumen LIMIT 5', engine)


<details>
<summary>👀 Ver solución</summary>

```python
# Análisis 1
q3 = df[(df['fecha'] >= '2023-07-01') & (df['fecha'] < '2023-10-01')]
print(f'  Registros Q3: {len(q3)}')
print(q3.groupby('categoria')['ingreso'].sum().sort_values(ascending=False))

# Análisis 2
mensual = df.groupby('mes')['ingreso'].sum()
variacion = mensual.pct_change().round(3) * 100
print(variacion.to_string())

# Análisis 3
resumen = df.groupby(['mes', 'categoria'])['ingreso'].sum().reset_index()
resumen['mes'] = resumen['mes'].astype(str)
resumen.to_sql('resumen_mensual', engine, if_exists='replace', index=False)
print('  Tabla exportada')

# Verificar
df_check = pd.read_sql('SELECT * FROM resumen_mensual LIMIT 5', engine)
print(df_check)
```
</details>

### 🚀 Proyecto final — Pipeline completo de análisis

Integrá todo lo aprendido en este módulo:

**Escenario:** Sos analista de datos en una empresa de e-commerce. Tenés que construir un pipeline completo.

**Paso 1 — Modelado con SQLAlchemy ORM:**
Creá los modelos para: `Cliente`, `Producto`, `Categoria`, `Pedido`, `DetallePedido`

**Paso 2 — Carga de datos:**
Insertá al menos 200 registros de pedidos con datos realistas usando `numpy` para generar variedad.

**Paso 3 — Análisis con Pandas:**
Usando `pd.read_sql()`, respondé:
1. ¿Cuál es el **ingreso total por mes**? ¿Hay estacionalidad?
2. ¿Qué **categoría de producto** genera más ingresos?
3. ¿Cuál es el **cliente con mayor lifetime value** (total gastado)?
4. ¿Qué productos tienen **baja rotación** (pocas unidades vendidas)?
5. Calculá el **ticket promedio** por cliente con `numpy`

**Paso 4 — Exportar resultados:**
Guardá cada análisis en una tabla separada dentro de la misma BD usando `to_sql()`.

In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, Column, Integer, String, Float, ForeignKey, text
from sqlalchemy.orm import DeclarativeBase, relationship, Session

# 🚀 Tu pipeline completo aquí:

# Paso 1: Modelos


# Paso 2: Datos


# Paso 3: Análisis


# Paso 4: Exportar


---
## ✅ Resumen del módulo

### Lo que aprendiste

| Herramienta | Conceptos clave |
|-------------|-----------------|
| `sqlite3` | Conexión, cursor, `execute`, `executemany`, FK, `with` context manager |
| SQLAlchemy Core | `create_engine`, `text()`, `engine.connect()` |
| SQLAlchemy ORM | `DeclarativeBase`, `Column`, `relationship`, `Session`, CRUD |
| Pandas + SQL | `pd.read_sql()`, `to_sql()`, pipeline BD → análisis |

### ¿Cuándo usar cada uno?

| Situación | Herramienta recomendada |
|-----------|------------------------|
| Scripts simples, aprendizaje | `sqlite3` puro |
| Aplicación Python con BD | SQLAlchemy ORM |
| Queries complejas y rendimiento | SQLAlchemy Core + `text()` |
| Análisis y reportes | `pd.read_sql()` + Pandas |
| ETL / pipelines de datos | SQLAlchemy + Pandas + `to_sql()` |

---

### 📚 Recursos recomendados
- [SQLAlchemy ORM — Tutorial oficial](https://docs.sqlalchemy.org/en/20/orm/quickstart.html)
- [Pandas — read_sql docs](https://pandas.pydata.org/docs/reference/api/pandas.read_sql.html)
- [SQLite Tutorial](https://www.sqlitetutorial.net/)
- [SQLAlchemy — Core Tutorial](https://docs.sqlalchemy.org/en/20/core/)